# 03_gurobi_ideal_model

This notebook builds and solves the implemented Part 1 benchmark MILP using the parameter files exported by Notebook 02.

Main outputs:
- optimal benchmark objective value
- facility-level expansion decisions
- ZIP-code and new-build solution summaries
- exported solution tables for reporting and visualization


In [1]:
# 03_gurobi_ideal_model

'''This notebook builds and solves the ideal child care expansion model using Gurobi.

Main tasks:
1. Load ideal model input tables
2. Build the MILP formulation
3. Solve the ideal model
4. Export solution tables for reporting and visualization'''

'This notebook builds and solves the ideal child care expansion model using Gurobi.\n\nMain tasks:\n1. Load ideal model input tables\n2. Build the MILP formulation\n3. Solve the ideal model\n4. Export solution tables for reporting and visualization'

## Setup and input loading

The first cells configure the benchmark model environment, check that the exported parameter files exist, and load them into memory for optimization.


In [2]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [3]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

In [4]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

IDEAL_DIR = PROJECT_ROOT / "data" / "processed" / "ideal"
RESULTS_DIR = PROJECT_ROOT / "results"
SOLUTIONS_DIR = RESULTS_DIR / "solutions"
LOG_DIR = RESULTS_DIR / "gurobi_logs"

SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IDEAL_DIR:", IDEAL_DIR)
print("SOLUTIONS_DIR:", SOLUTIONS_DIR)
print("LOG_DIR:", LOG_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
IDEAL_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal
SOLUTIONS_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/solutions
LOG_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/gurobi_logs


In [5]:
required_files = [
    "zipcode_demand_supply_ideal.csv",
    "facility_expansion_params_ideal.csv",
    "build_options_ideal.csv",
    "assumptions_ideal.csv"
]

for f in required_files:
    path = IDEAL_DIR / f
    print(f, "exists ->", path.exists())

zipcode_demand_supply_ideal.csv exists -> True
facility_expansion_params_ideal.csv exists -> True
build_options_ideal.csv exists -> True
assumptions_ideal.csv exists -> True


In [6]:
zip_df = pd.read_csv(IDEAL_DIR / "zipcode_demand_supply_ideal.csv")
fac_df = pd.read_csv(IDEAL_DIR / "facility_expansion_params_ideal.csv")
build_df = pd.read_csv(IDEAL_DIR / "build_options_ideal.csv")
assumptions_df = pd.read_csv(IDEAL_DIR / "assumptions_ideal.csv")

In [7]:
print("zipcode_demand_supply_ideal:", zip_df.shape)
display(zip_df.head())

print("facility_expansion_params_ideal:", fac_df.shape)
display(fac_df.head())

print("build_options_ideal:", build_df.shape)
display(build_df)

zipcode_demand_supply_ideal: (311, 23)


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1


facility_expansion_params_ideal: (7712, 12)


,facility_id,facility_name,zipcode,program_type,facility_status,n_f,b_f,U_f,cslot_f,alpha_big_f,x1_cap,x2_cap
0,229433,YMCA of Greater NY @Virtual Y PS 33,10001,SACC,Registration,88.0,0.0,105.6,250,427.272727,88.0,17.6
1,292419,The Hudson Guild @26th Street,10001,SACC,Registration,79.0,0.0,94.8,250,453.164557,79.0,15.8
2,350076,GRAMMAS HANDS,10001,FDC,Registration,8.0,6.0,9.6,350,2700.000000,8.0,1.6
3,661697,Chelsea Little Angels Day Care,10001,GFDC,License,16.0,12.0,19.2,350,1450.000000,16.0,3.2
4,827488,"Davis, Milinaire",10001,FDC,Registration,8.0,6.0,9.6,350,2700.000000,8.0,1.6


build_options_ideal: (3, 4)


,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost
0,small,100,50,65000
1,medium,200,100,95000
2,large,400,200,115000


## Model design

The benchmark model has two intervention levers:
- expand existing facilities already operating in each ZIP code
- build new facilities by size within each ZIP code

The corrected Part 1 implementation uses the following facility-level variables:
- `x_f`: total expansion slots added at facility `f`
- `u_f`: under-5 slots assigned within that expansion
- `delta_f`: binary indicator equal to 1 when `x_f \ge n_f`
- `w_f`: modeled expansion cost at facility `f`

At the ZIP-code level, the model uses:
- `y_{zs}`: number of new facilities of size `s` built in ZIP code `z`
- `g_{zs}`: under-5 slots assigned within those new builds

The `delta_f` and `w_f` variables are what make the corrected benchmark cost rule work. Once expansion reaches the current facility size, the whole expansion is priced under the higher-cost regime rather than additively by blocks.

In [8]:
#Final type cleanup before modeling
# Standardize zipcode and size keys as strings
zip_df["zipcode"] = zip_df["zipcode"].astype(str).str.zfill(5)
fac_df["zipcode"] = fac_df["zipcode"].astype(str).str.zfill(5)
build_df["size"] = build_df["size"].astype(str)

# Integer-safe modeling cap for total expansion
fac_df["U_f_int"] = np.floor(fac_df["U_f"]).astype(int)

# Optional: make current capacities numeric
numeric_cols_zip = [
    "req_total", "req_under5", "existing_total", "existing_under5",
    "gap_total", "gap_under5"
]
for c in numeric_cols_zip:
    if c in zip_df.columns:
        zip_df[c] = pd.to_numeric(zip_df[c], errors="coerce").fillna(0)

numeric_cols_fac = [
    "n_f", "b_f", "U_f", "cslot_f", "alpha_big_f", "U_f_int"
]
for c in numeric_cols_fac:
    if c in fac_df.columns:
        fac_df[c] = pd.to_numeric(fac_df[c], errors="coerce").fillna(0)

display(fac_df[["facility_id", "zipcode", "n_f", "U_f", "U_f_int", "cslot_f", "alpha_big_f"]].head())

,facility_id,zipcode,n_f,U_f,U_f_int,cslot_f,alpha_big_f
0,229433,10001,88.0,105.6,105,250,427.272727
1,292419,10001,79.0,94.8,94,250,453.164557
2,350076,10001,8.0,9.6,9,350,2700.000000
3,661697,10001,16.0,19.2,19,350,1450.000000
4,827488,10001,8.0,9.6,9,350,2700.000000


In [9]:
#Build sets and parameter dictionaries
Z = zip_df["zipcode"].tolist()
F = fac_df["facility_id"].tolist()
S = build_df["size"].tolist()

# Zipcode parameters
req_total = dict(zip(zip_df["zipcode"], zip_df["req_total"]))
req_under5 = dict(zip(zip_df["zipcode"], zip_df["req_under5"]))
existing_total = dict(zip(zip_df["zipcode"], zip_df["existing_total"]))
existing_under5 = dict(zip(zip_df["zipcode"], zip_df["existing_under5"]))

# Facility parameters
zip_of_f = dict(zip(fac_df["facility_id"], fac_df["zipcode"]))
n_f = dict(zip(fac_df["facility_id"], fac_df["n_f"]))
b_f = dict(zip(fac_df["facility_id"], fac_df["b_f"]))
U_f_dict = dict(zip(fac_df["facility_id"], fac_df["U_f_int"]))
cslot_f = dict(zip(fac_df["facility_id"], fac_df["cslot_f"]))
alpha_big_f = dict(zip(fac_df["facility_id"], fac_df["alpha_big_f"]))

# Build-option parameters
build_total = dict(zip(build_df["size"], build_df["new_total_capacity"]))
build_under5_max = dict(zip(build_df["size"], build_df["new_under5_capacity_max"]))
build_cost = dict(zip(build_df["size"], build_df["fixed_build_cost"]))

# Facilities by zipcode
fac_by_zip = fac_df.groupby("zipcode")["facility_id"].apply(list).to_dict()

print("Number of zipcodes:", len(Z))
print("Number of facilities:", len(F))
print("Build sizes:", S)

Number of zipcodes: 311
Number of facilities: 7712
Build sizes: ['small', 'medium', 'large']


In [10]:
print("Any zipcode missing req_total ? -", zip_df["req_total"].isna().sum())
print("Any zipcode missing req_under5 ? -", zip_df["req_under5"].isna().sum())
print("Any facility with negative total expansion cap ? -", (fac_df["U_f_int"] < 0).any())
print("Any facility with negative cost coeff ? -", (fac_df["cslot_f"] < 0).any() or (fac_df["alpha_big_f"] < 0).any())

Any zipcode missing req_total ? - 0
Any zipcode missing req_under5 ? - 0
Any facility with negative total expansion cap ? - False
Any facility with negative cost coeff ? - False


In [11]:
#Solver parameters
log_file = LOG_DIR / "ideal_model.log"
model = gp.Model("ideal_childcare")
model.Params.LogFile = str(log_file)
model.Params.MIPGap = 0.001
model.Params.TimeLimit = 600
model.Params.Presolve = 2
model.Params.Threads = 0

Set parameter Username


Set parameter LicenseID to value 2773994


Academic license - for non-commercial use only - expires 2027-02-02


Set parameter LogFile to value "/Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/gurobi_logs/ideal_model.log"


Set parameter MIPGap to value 0.001


Set parameter TimeLimit to value 600


Set parameter Presolve to value 2


Set parameter Threads to value 0


In [12]:
#Add decision variables
# Expansion variables for existing facilities
x = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x")           # total expansion
u = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="u")           # under-5 slots among expansion
w = model.addVars(F, vtype=GRB.CONTINUOUS, lb=0, name="w")        # expansion cost

delta = model.addVars(F, vtype=GRB.BINARY, name="delta")          # 1 if x_f >= n_f

# New build variables by zipcode and size
y = model.addVars(Z, S, vtype=GRB.INTEGER, lb=0, name="y")        # number of facilities built
g = model.addVars(Z, S, vtype=GRB.INTEGER, lb=0, name="g")        # under-5 slots in new builds

print("Variables added.")

Variables added.


In [13]:
#Facility-level constraints
for f in F:
    # overall expansion upper bound
    model.addConstr(x[f] <= U_f_dict[f], name=f"x_cap_{f}")

    # under-5 expansion cannot exceed total added slots
    model.addConstr(u[f] <= x[f], name=f"u_total_link_{f}")

    # binary threshold logic:
    # delta[f] = 0  => x[f] <= n_f - 1
    # delta[f] = 1  => x[f] >= n_f
    M_f = U_f_dict[f]
    n_val = int(np.floor(n_f[f]))

    model.addConstr(
        x[f] <= (n_val - 1) + M_f * delta[f],
        name=f"threshold_up_{f}"
    )

    model.addConstr(
        x[f] >= n_val * delta[f],
        name=f"threshold_low_{f}"
    )

    # Big-M cost linking so the whole expansion is priced under one regime
    M_cost = max(cslot_f[f], alpha_big_f[f]) * max(U_f_dict[f], 1)

    # If delta[f] = 0, then w[f] = cslot_f[f] * x[f]
    model.addConstr(
        w[f] >= cslot_f[f] * x[f] - M_cost * delta[f],
        name=f"cost_low_lb_{f}"
    )
    model.addConstr(
        w[f] <= cslot_f[f] * x[f] + M_cost * delta[f],
        name=f"cost_low_ub_{f}"
    )

    # If delta[f] = 1, then w[f] = alpha_big_f[f] * x[f]
    model.addConstr(
        w[f] >= alpha_big_f[f] * x[f] - M_cost * (1 - delta[f]),
        name=f"cost_high_lb_{f}"
    )
    model.addConstr(
        w[f] <= alpha_big_f[f] * x[f] + M_cost * (1 - delta[f]),
        name=f"cost_high_ub_{f}"
    )

In [14]:
#Zipcode-level total capacity constraints
for z in Z:
    model.addConstr(
        existing_total[z]
        + gp.quicksum(x[f] for f in fac_by_zip.get(z, []))
        + gp.quicksum(build_total[s] * y[z, s] for s in S)
        >= req_total[z],
        name=f"total_req_{z}"
    )

In [15]:
#Zipcode-level under-5 constraints
for z in Z:
    model.addConstr(
        existing_under5[z]
        + gp.quicksum(u[f] for f in fac_by_zip.get(z, []))
        + gp.quicksum(g[z, s] for s in S)
        >= req_under5[z],
        name=f"under5_req_{z}"
    )

In [16]:
#Under-5 capacity limits for new facilities
for z in Z:
    for s in S:
        model.addConstr(
            g[z, s] <= build_under5_max[s] * y[z, s],
            name=f"new_under5_cap_{z}_{s}"
        )

## Objective function

The benchmark objective minimizes three funding components:
- expansion cost at existing facilities
- fixed construction cost for new facilities
- equipment cost of 100 dollars for each new under-5 slot

Because the corrected benchmark model prices the entire expansion under one regime once `x_f \ge n_f`, the objective uses the auxiliary cost variable `w_f` instead of the earlier additive `x1/x2` block formulation.

In [17]:
#Set objective
expansion_cost_expr = gp.quicksum(
    w[f] for f in F
)

new_build_cost_expr = gp.quicksum(
    build_cost[s] * y[z, s]
    for z in Z for s in S
)

under5_equipment_cost_expr = 100 * (
    gp.quicksum(u[f] for f in F) +
    gp.quicksum(g[z, s] for z in Z for s in S)
)

model.setObjective(
    expansion_cost_expr + new_build_cost_expr + under5_equipment_cost_expr,
    GRB.MINIMIZE
)

In [18]:
#Model size check
model.update()

print("Number of variables:", model.NumVars)
print("Number of constraints:", model.NumConstrs)

Number of variables: 32714
Number of constraints: 63251


## Solve and status checks

These cells solve the benchmark MILP, interpret the returned model status, and report the core run-time diagnostics used later in the report.


In [19]:
#Optimize
start_time = time.time()
model.optimize()
solve_time = time.time() - start_time

print(f"Solve time: {solve_time:.2f} seconds")
print("Model status code:", model.Status)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D2128)


CPU model: Apple M4 Pro


Thread count: 12 physical cores, 12 logical processors, using up to 12 threads


Non-default parameters:


TimeLimit  600


...

Thread count was 12 (of 12 available processors)


Solution count 4: 1.26669e+08 1.26784e+08 1.27457e+08 3.11912e+08 


Optimal solution found (tolerance 1.00e-03)


Best objective 1.266689407981e+08, best bound 1.266369664395e+08, gap 0.0252%


Solve time: 2.42 seconds
Model status code: 2


In [20]:
#Interpret solve status
status_map = {
    GRB.OPTIMAL: "OPTIMAL",
    GRB.TIME_LIMIT: "TIME_LIMIT",
    GRB.INFEASIBLE: "INFEASIBLE",
    GRB.INF_OR_UNBD: "INF_OR_UNBD",
    GRB.UNBOUNDED: "UNBOUNDED",
    GRB.INTERRUPTED: "INTERRUPTED"
}

print("Solve status:", status_map.get(model.Status, f"OTHER ({model.Status})"))

Solve status: OPTIMAL


In [21]:
#Optional infeasibility diagnosis
if model.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    model.computeIIS()
    iis_path = SOLUTIONS_DIR / "ideal_model.ilp"
    model.write(str(iis_path))
    print("IIS written to:", iis_path)

In [22]:
#Only continue if a solution exists
has_solution = model.SolCount > 0
print("Has solution:", has_solution)

Has solution: True


## Solution tables and exports

After the solve, the notebook assembles the benchmark solution tables used in the report:
- objective breakdowns
- facility-level expansions and selected cost regimes
- ZIP-code summaries
- new-build decisions
- run metadata and summary statistics

The exported facility table now reports `x_added`, `delta_triggered`, and `selected_cost_regime`, which are the benchmark fields used downstream in Notebook 04 and the report tables.

In [23]:
#Objective value and key cost components
if has_solution:
    objective_value = model.ObjVal
    expansion_cost_value = expansion_cost_expr.getValue()
    new_build_cost_value = new_build_cost_expr.getValue()
    under5_equipment_cost_value = under5_equipment_cost_expr.getValue()

    print("Objective value:", objective_value)
    print("Expansion cost:", expansion_cost_value)
    print("New build cost:", new_build_cost_value)
    print("Under-5 equipment cost:", under5_equipment_cost_value)

Objective value: 126668940.79807301
Expansion cost: 54716840.79807301
New build cost: 44180000.0
Under-5 equipment cost: 27772100.0


In [24]:
#Facility expansion solution table
if has_solution:
    facility_solution = fac_df[[
        "facility_id", "facility_name", "zipcode", "program_type",
        "n_f", "b_f", "U_f", "U_f_int", "cslot_f", "alpha_big_f"
    ]].copy()

    facility_solution["x_added"] = facility_solution["facility_id"].map(lambda f: x[f].X)
    facility_solution["x_total_added"] = facility_solution["x_added"]  # compatibility alias for downstream summaries
    facility_solution["u_under5_added"] = facility_solution["facility_id"].map(lambda f: u[f].X)
    facility_solution["delta_triggered"] = facility_solution["facility_id"].map(lambda f: int(round(delta[f].X)))
    facility_solution["selected_cost_regime"] = np.where(
        facility_solution["delta_triggered"] == 1,
        "high_cost_rule",
        "low_cost_rule"
    )
    facility_solution["expansion_cost"] = facility_solution["facility_id"].map(lambda f: w[f].X)

    facility_solution["under5_equipment_cost_expansion"] = 100 * facility_solution["u_under5_added"]
    facility_solution["total_facility_related_cost"] = (
        facility_solution["expansion_cost"] +
        facility_solution["under5_equipment_cost_expansion"]
    )

    facility_solution = facility_solution.sort_values(
        ["x_added", "u_under5_added"],
        ascending=False
    ).reset_index(drop=True)

    display(facility_solution.head(20))

,facility_id,facility_name,zipcode,program_type,n_f,b_f,U_f,U_f_int,cslot_f,alpha_big_f,x_added,x_total_added,u_under5_added,delta_triggered,selected_cost_regime,expansion_cost,under5_equipment_cost_expansion,total_facility_related_cost
0,866851,"Imogen Roche Foundation, Inc.",10016,SACC,480.0,0.0,500.0,500,250,241.666667,500.0,500.0,500.0,1,high_cost_rule,120833.333333,50000.0,170833.333333
1,905401,"87 Afterschool, Inc.",10024,SACC,540.0,0.0,500.0,500,250,237.037037,500.0,500.0,500.0,0,low_cost_rule,125000.000000,50000.0,175000.000000
2,886512,WildArts Inc.,10025,SACC,560.0,0.0,500.0,500,250,235.714286,500.0,500.0,500.0,0,low_cost_rule,125000.000000,50000.0,175000.000000
3,709901,"Harlem Children's Zone, Inc. Promise Academy 1...",10027,SACC,443.0,0.0,500.0,500,250,245.146727,500.0,500.0,500.0,1,high_cost_rule,122573.363431,50000.0,172573.363431
4,73664,The Children's Aid Society @IS 90,10032,SACC,486.0,0.0,500.0,500,250,241.152263,500.0,500.0,500.0,1,high_cost_rule,120576.131687,50000.0,170576.131687
5,889892,"Imogen Roche Foundation, Inc.",10065,SACC,469.0,0.0,500.0,500,250,242.643923,500.0,500.0,500.0,1,high_cost_rule,121321.961620,50000.0,171321.961620
6,661155,"New York Center for Interpersonal Development,...",10304,SACC,437.0,0.0,500.0,500,250,245.766590,500.0,500.0,500.0,1,high_cost_rule,122883.295195,50000.0,172883.295195
7,481959,"New York Center for Interpersonal Development,...",10314,SACC,890.0,0.0,500.0,500,250,222.471910,500.0,500.0,500.0,0,low_cost_rule,125000.000000,50000.0,175000.000000
8,637483,"Directions for Our Youth, Inc.",10453,SACC,464.0,0.0,500.0,500,250,243.103448,500.0,500.0,500.0,1,high_cost_rule,121551.724138,50000.0,171551.724138
9,479327,SoBRO,10460,SACC,500.0,0.0,500.0,500,250,240.000000,500.0,500.0,500.0,1,high_cost_rule,120000.000000,50000.0,170000.000000


In [25]:
#New build solution table
if has_solution:
    rows = []
    for z in Z:
        for s in S:
            y_val = y[z, s].X
            g_val = g[z, s].X
            if y_val > 1e-6 or g_val > 1e-6:
                rows.append({
                    "zipcode": z,
                    "size": s,
                    "num_new_facilities": y_val,
                    "new_total_capacity_per_facility": build_total[s],
                    "new_under5_capacity_max_per_facility": build_under5_max[s],
                    "new_under5_assigned": g_val,
                    "fixed_build_cost_per_facility": build_cost[s],
                    "fixed_build_cost_total": build_cost[s] * y_val,
                    "under5_equipment_cost_newbuild": 100 * g_val,
                    "total_newbuild_related_cost": build_cost[s] * y_val + 100 * g_val
                })

    new_build_solution = pd.DataFrame(rows)

    if len(new_build_solution) > 0:
        new_build_solution = new_build_solution.sort_values(
            ["num_new_facilities", "zipcode", "size"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

    display(new_build_solution.head(20))

,zipcode,size,num_new_facilities,new_total_capacity_per_facility,new_under5_capacity_max_per_facility,new_under5_assigned,fixed_build_cost_per_facility,fixed_build_cost_total,under5_equipment_cost_newbuild,total_newbuild_related_cost
0,11219,large,22.0,400,200,4400.0,115000,2530000.0,440000.0,2970000.0
1,11230,large,21.0,400,200,4200.0,115000,2415000.0,420000.0,2835000.0
2,11223,large,14.0,400,200,2800.0,115000,1610000.0,280000.0,1890000.0
3,11249,large,14.0,400,200,2800.0,115000,1610000.0,280000.0,1890000.0
4,10314,large,11.0,400,200,2200.0,115000,1265000.0,220000.0,1485000.0
5,11205,large,11.0,400,200,2200.0,115000,1265000.0,220000.0,1485000.0
6,11220,large,11.0,400,200,1766.0,115000,1265000.0,176600.0,1441600.0
7,11368,large,9.0,400,200,1800.0,115000,1035000.0,180000.0,1215000.0
8,10024,large,8.0,400,200,1600.0,115000,920000.0,160000.0,1080000.0
9,10128,large,8.0,400,200,1600.0,115000,920000.0,160000.0,1080000.0


In [26]:
#Zipcode-level solution summary
if has_solution:
    # Expansion aggregated by zipcode
    exp_zip = (
        facility_solution.groupby("zipcode")
        .agg(
            added_total_expansion=("x_added", "sum"),
            added_under5_expansion=("u_under5_added", "sum"),
            expansion_cost=("expansion_cost", "sum"),
            under5_equipment_cost_expansion=("under5_equipment_cost_expansion", "sum")
        )
        .reset_index()
    )

    # New builds aggregated by zipcode
    if len(new_build_solution) > 0:
        new_zip = (
            new_build_solution.groupby("zipcode")
            .agg(
                num_new_facilities_total=("num_new_facilities", "sum"),
                added_under5_newbuild=("new_under5_assigned", "sum"),
                fixed_build_cost_total=("fixed_build_cost_total", "sum"),
                under5_equipment_cost_newbuild=("under5_equipment_cost_newbuild", "sum")
            )
            .reset_index()
        )

        # total slots from new builds
        tmp = new_build_solution.copy()
        tmp["added_total_newbuild"] = tmp["num_new_facilities"] * tmp["new_total_capacity_per_facility"]
        new_zip_total = tmp.groupby("zipcode")["added_total_newbuild"].sum().reset_index()

        new_zip = new_zip.merge(new_zip_total, on="zipcode", how="left")
    else:
        new_zip = pd.DataFrame(columns=[
            "zipcode", "num_new_facilities_total", "added_under5_newbuild",
            "fixed_build_cost_total", "under5_equipment_cost_newbuild", "added_total_newbuild"
        ])

    zipcode_solution = zip_df.copy()

    zipcode_solution = zipcode_solution.merge(exp_zip, on="zipcode", how="left")
    zipcode_solution = zipcode_solution.merge(new_zip, on="zipcode", how="left")

    fill_cols = [
        "added_total_expansion", "added_under5_expansion", "expansion_cost",
        "under5_equipment_cost_expansion", "num_new_facilities_total",
        "added_under5_newbuild", "fixed_build_cost_total",
        "under5_equipment_cost_newbuild", "added_total_newbuild"
    ]
    for c in fill_cols:
        zipcode_solution[c] = zipcode_solution[c].fillna(0)

    zipcode_solution["final_total_capacity"] = (
        zipcode_solution["existing_total"]
        + zipcode_solution["added_total_expansion"]
        + zipcode_solution["added_total_newbuild"]
    )

    zipcode_solution["final_under5_capacity"] = (
        zipcode_solution["existing_under5"]
        + zipcode_solution["added_under5_expansion"]
        + zipcode_solution["added_under5_newbuild"]
    )

    zipcode_solution["meets_total_requirement_after"] = (
        zipcode_solution["final_total_capacity"] >= zipcode_solution["req_total"]
    ).astype(int)

    zipcode_solution["meets_under5_requirement_after"] = (
        zipcode_solution["final_under5_capacity"] >= zipcode_solution["req_under5"]
    ).astype(int)

    zipcode_solution["all_requirements_met_after"] = (
        (zipcode_solution["meets_total_requirement_after"] == 1) &
        (zipcode_solution["meets_under5_requirement_after"] == 1)
    ).astype(int)

    zipcode_solution["total_capacity_surplus_after"] = (
        zipcode_solution["final_total_capacity"] - zipcode_solution["req_total"]
    )

    zipcode_solution["under5_capacity_surplus_after"] = (
        zipcode_solution["final_under5_capacity"] - zipcode_solution["req_under5"]
    )

    display(zipcode_solution.head(20))

,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before,added_total_expansion,added_under5_expansion,expansion_cost,under5_equipment_cost_expansion,num_new_facilities_total,added_under5_newbuild,fixed_build_cost_total,under5_equipment_cost_newbuild,added_total_newbuild,final_total_capacity,final_under5_capacity,meets_total_requirement_after,meets_under5_requirement_after,all_requirements_met_after,total_capacity_surplus_after,under5_capacity_surplus_after
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1,472.0,472.0,118000.000000,47200.0,0.0,0.0,0.0,0.0,0.0,1081.0,496.0,1,1,1,414.0,0.0
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1,1230.0,1230.0,307500.000000,123000.0,0.0,0.0,0.0,0.0,0.0,5954.0,1428.0,1,1,1,2560.0,0.0
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1,962.0,960.0,232400.000000,96000.0,0.0,0.0,0.0,0.0,0.0,2957.0,960.0,1,1,1,1973.0,0.0
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1,289.0,289.0,80920.000000,28900.0,0.0,0.0,0.0,0.0,0.0,552.0,289.0,1,1,1,320.0,0.0
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1,23.0,23.0,6900.000000,2300.0,2.0,300.0,210000.0,30000.0,600.0,662.0,323.0,1,1,1,260.0,0.0
5,10006,128.0,133.0,261.0,0.631692,126377.118644,0,0,0,0,0,1,130.500000,131,86,156.0,0.0,1.0,0.0,86.0,0,1,1,86.0,86.0,21500.000000,8600.0,0.0,0.0,0.0,0.0,0.0,242.0,86.0,1,1,1,111.0,0.0
6,10007,605.0,596.0,1201.0,0.528910,138853.904282,0,0,0,0,0,0,400.333333,401,404,284.0,0.0,3.0,117.0,404.0,1,1,1,204.0,204.0,51000.000000,20400.0,1.0,200.0,115000.0,20000.0,400.0,888.0,404.0,1,1,1,487.0,0.0
7,10008,0.0,0.0,0.0,0.476652,61197.281283,1,1,1,1,1,0,0.000000,0,0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,0.0,0.0
8,10009,1896.0,2575.0,4471.0,0.514567,77133.233533,0,0,0,0,0,0,1490.333333,1491,1264,1784.0,106.0,23.0,0.0,1158.0,0,1,1,1158.0,1158.0,289500.000000,115800.0,0.0,0.0,0.0,0.0,0.0,2942.0,1264.0,1,1,1,1451.0,0.0
9,10010,1422.0,2066.0,3488.0,0.492749,116272.698810,0,0,0,0,0,0,1162.666667,1163,948,234.0,0.0,3.0,929.0,948.0,1,1,1,148.0,148.0,37450.000000,14800.0,4.0,800.0,460000.0,80000.0,1600.0,1982.0,948.0,1,1,1,819.0,0.0


In [27]:
#Objective breakdown table
if has_solution:
    objective_breakdown = pd.DataFrame([
        {"component": "expansion_cost", "value": expansion_cost_value},
        {"component": "new_build_cost", "value": new_build_cost_value},
        {"component": "under5_equipment_cost", "value": under5_equipment_cost_value},
        {"component": "total_objective", "value": objective_value}
    ])

    display(objective_breakdown)

,component,value
0,expansion_cost,5.471684e+07
1,new_build_cost,4.418000e+07
2,under5_equipment_cost,2.777210e+07
3,total_objective,1.266689e+08


In [28]:
#Run metadata table
if has_solution:
    run_metadata = pd.DataFrame([
        {"metric": "model_status_code", "value": model.Status},
        {"metric": "model_status_text", "value": status_map.get(model.Status, str(model.Status))},
        {"metric": "objective_value", "value": model.ObjVal},
        {"metric": "best_bound", "value": model.ObjBound},
        {"metric": "mip_gap", "value": model.MIPGap if model.IsMIP else 0},
        {"metric": "solve_time_seconds", "value": solve_time},
        {"metric": "num_variables", "value": model.NumVars},
        {"metric": "num_constraints", "value": model.NumConstrs},
        {"metric": "num_solutions", "value": model.SolCount}
    ])

    display(run_metadata)

,metric,value
0,model_status_code,2
1,model_status_text,OPTIMAL
2,objective_value,126668940.798073
3,best_bound,126636966.439482
4,mip_gap,0.000252
5,solve_time_seconds,2.416418
6,num_variables,32714
7,num_constraints,63251
8,num_solutions,4


In [29]:
#Quick management summary for the report
if has_solution:
    total_added_expansion = facility_solution["x_added"].sum()
    total_added_under5_expansion = facility_solution["u_under5_added"].sum()

    if len(new_build_solution) > 0:
        total_new_facilities = new_build_solution["num_new_facilities"].sum()
        total_added_newbuild = (
            new_build_solution["num_new_facilities"] *
            new_build_solution["new_total_capacity_per_facility"]
        ).sum()
        total_added_under5_newbuild = new_build_solution["new_under5_assigned"].sum()
    else:
        total_new_facilities = 0
        total_added_newbuild = 0
        total_added_under5_newbuild = 0

    summary_stats = pd.DataFrame([
        {"metric": "total_added_expansion_slots", "value": total_added_expansion},
        {"metric": "total_added_under5_expansion_slots", "value": total_added_under5_expansion},
        {"metric": "total_new_facilities_built", "value": total_new_facilities},
        {"metric": "total_added_newbuild_slots", "value": total_added_newbuild},
        {"metric": "total_added_under5_newbuild_slots", "value": total_added_under5_newbuild},
        {"metric": "facilities_in_high_cost_regime", "value": facility_solution["delta_triggered"].sum()},
        {"metric": "zipcodes_meeting_all_requirements_after", "value": zipcode_solution["all_requirements_met_after"].sum()},
        {"metric": "zipcodes_not_meeting_all_requirements_after", "value": (zipcode_solution["all_requirements_met_after"] == 0).sum()}
    ])

    display(summary_stats)

,metric,value
0,total_added_expansion_slots,206364.0
1,total_added_under5_expansion_slots,202938.0
2,total_new_facilities_built,386.0
3,total_added_newbuild_slots,152900.0
4,total_added_under5_newbuild_slots,74783.0
5,facilities_in_high_cost_regime,135.0
6,zipcodes_meeting_all_requirements_after,311.0
7,zipcodes_not_meeting_all_requirements_after,0.0


In [30]:
if has_solution:
    print("Top 15 facilities by added expansion:")
    display(
        facility_solution.loc[facility_solution["x_added"] > 0, [
            "facility_id", "facility_name", "zipcode",
            "n_f", "x_added", "u_under5_added",
            "delta_triggered", "selected_cost_regime", "expansion_cost"
        ]].head(15)
    )

    if len(new_build_solution) > 0:
        print("Top zipcode-size new build decisions:")
        display(
            new_build_solution.loc[new_build_solution["num_new_facilities"] > 0].head(15)
        )

    print("Top 15 zipcodes by final added capacity:")
    display(
        zipcode_solution.assign(
            total_added=lambda d: d["added_total_expansion"] + d["added_total_newbuild"]
        ).sort_values("total_added", ascending=False)[[
            "zipcode", "total_added", "added_total_expansion", "added_total_newbuild",
            "added_under5_expansion", "added_under5_newbuild",
            "existing_total", "req_total", "final_total_capacity"
        ]].head(15)
    )

Top 15 facilities by added expansion:


,facility_id,facility_name,zipcode,n_f,x_added,u_under5_added,delta_triggered,selected_cost_regime,expansion_cost
0,866851,"Imogen Roche Foundation, Inc.",10016,480.0,500.0,500.0,1,high_cost_rule,120833.333333
1,905401,"87 Afterschool, Inc.",10024,540.0,500.0,500.0,0,low_cost_rule,125000.000000
2,886512,WildArts Inc.,10025,560.0,500.0,500.0,0,low_cost_rule,125000.000000
3,709901,"Harlem Children's Zone, Inc. Promise Academy 1...",10027,443.0,500.0,500.0,1,high_cost_rule,122573.363431
4,73664,The Children's Aid Society @IS 90,10032,486.0,500.0,500.0,1,high_cost_rule,120576.131687
5,889892,"Imogen Roche Foundation, Inc.",10065,469.0,500.0,500.0,1,high_cost_rule,121321.961620
6,661155,"New York Center for Interpersonal Development,...",10304,437.0,500.0,500.0,1,high_cost_rule,122883.295195
7,481959,"New York Center for Interpersonal Development,...",10314,890.0,500.0,500.0,0,low_cost_rule,125000.000000
8,637483,"Directions for Our Youth, Inc.",10453,464.0,500.0,500.0,1,high_cost_rule,121551.724138
9,479327,SoBRO,10460,500.0,500.0,500.0,1,high_cost_rule,120000.000000


Top zipcode-size new build decisions:


,zipcode,size,num_new_facilities,new_total_capacity_per_facility,new_under5_capacity_max_per_facility,new_under5_assigned,fixed_build_cost_per_facility,fixed_build_cost_total,under5_equipment_cost_newbuild,total_newbuild_related_cost
0,11219,large,22.0,400,200,4400.0,115000,2530000.0,440000.0,2970000.0
1,11230,large,21.0,400,200,4200.0,115000,2415000.0,420000.0,2835000.0
2,11223,large,14.0,400,200,2800.0,115000,1610000.0,280000.0,1890000.0
3,11249,large,14.0,400,200,2800.0,115000,1610000.0,280000.0,1890000.0
4,10314,large,11.0,400,200,2200.0,115000,1265000.0,220000.0,1485000.0
5,11205,large,11.0,400,200,2200.0,115000,1265000.0,220000.0,1485000.0
6,11220,large,11.0,400,200,1766.0,115000,1265000.0,176600.0,1441600.0
7,11368,large,9.0,400,200,1800.0,115000,1035000.0,180000.0,1215000.0
8,10024,large,8.0,400,200,1600.0,115000,920000.0,160000.0,1080000.0
9,10128,large,8.0,400,200,1600.0,115000,920000.0,160000.0,1080000.0


Top 15 zipcodes by final added capacity:


,zipcode,total_added,added_total_expansion,added_total_newbuild,added_under5_expansion,added_under5_newbuild,existing_total,req_total,final_total_capacity
213,11219,11472.0,2672.0,8800.0,2672.0,4400.0,2595.0,13374,14067.0
223,11230,9167.0,767.0,8400.0,767.0,4200.0,824.0,5803,9991.0
217,11223,7084.0,1484.0,5600.0,1484.0,2800.0,1448.0,7741,8532.0
258,11368,6564.0,2964.0,3600.0,2961.0,1800.0,3057.0,9621,9621.0
214,11220,6093.0,1693.0,4400.0,1693.0,1766.0,2251.0,8344,8344.0
238,11249,5887.0,287.0,5600.0,287.0,2800.0,315.0,3204,6202.0
200,11206,5644.0,3244.0,2400.0,3244.0,1200.0,3233.0,8618,8877.0
198,11204,5574.0,3174.0,2400.0,3174.0,1200.0,3192.0,8683,8766.0
159,10314,5557.0,1157.0,4400.0,1157.0,2200.0,1654.0,4624,7211.0
199,11205,5232.0,832.0,4400.0,832.0,2200.0,918.0,3711,6150.0


In [31]:
if has_solution:
    facility_solution.to_csv(SOLUTIONS_DIR / "facility_solution_ideal.csv", index=False)
    zipcode_solution.to_csv(SOLUTIONS_DIR / "zipcode_solution_ideal.csv", index=False)
    objective_breakdown.to_csv(SOLUTIONS_DIR / "objective_breakdown_ideal.csv", index=False)
    run_metadata.to_csv(SOLUTIONS_DIR / "run_metadata_ideal.csv", index=False)
    summary_stats.to_csv(SOLUTIONS_DIR / "summary_stats_ideal.csv", index=False)

    if len(new_build_solution) > 0:
        new_build_solution.to_csv(SOLUTIONS_DIR / "new_build_solution_ideal.csv", index=False)
    else:
        pd.DataFrame(columns=[
            "zipcode", "size", "num_new_facilities", "new_total_capacity_per_facility",
            "new_under5_capacity_max_per_facility", "new_under5_assigned",
            "fixed_build_cost_per_facility", "fixed_build_cost_total",
            "under5_equipment_cost_newbuild", "total_newbuild_related_cost"
        ]).to_csv(SOLUTIONS_DIR / "new_build_solution_ideal.csv", index=False)

    print("Solution files saved to:", SOLUTIONS_DIR)

Solution files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/solutions
